In [1]:
# Setup + Q1
import sys
from pathlib import Path

import numpy as np

EMBED_DIR = (Path.cwd() / ".." / "embed").resolve()
MODEL_DIR = EMBED_DIR / "models" / "Xenova" / "all-MiniLM-L6-v2"
sys.path.insert(0, str(EMBED_DIR))

from embedder import Embedder

model = Embedder(path=MODEL_DIR)

query = "How does approximate nearest neighbor search work?"
v = model.encode(query)

print(f"Q1 shape: {v.shape}")
print(f"Q1 v[0]: {v[0]:.2f}")  # form: -0.02


Q1 shape: (384,)
Q1 v[0]: -0.02


In [2]:
# Load documents
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
print(f"documents: {len(documents)}")


documents: 72


In [3]:
# Q2
# Official: 07-sqlitesearch-vector.md content vs Q1 query vector
target = next(
    d for d in documents
    if d["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"
)
v_doc = model.encode(target["content"])
cosine_sim = float(np.dot(v, v_doc))
print(f"Q2 cosine similarity: {cosine_sim:.2f}")  # form: 0.37


Q2 cosine similarity: 0.36


In [4]:
# Q3
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
chunk_contents = [c["content"] for c in chunks]

parts = []
for i in range(0, len(chunk_contents), 50):
    parts.append(model.encode_batch(chunk_contents[i : i + 50]))
X = np.vstack(parts)

scores = X.dot(v)
idx = int(np.argmax(scores))
print(f"Q3 score: {scores[idx]:.4f}")
print(f"Q3 filename: {chunks[idx]['filename']}")


Q3 score: 0.6489
Q3 filename: 02-vector-search/lessons/07-sqlitesearch-vector.md


In [5]:
# Q4
from minsearch import VectorSearch

ms_vector = VectorSearch(keyword_fields=["filename"])
ms_vector.fit(X, chunks)

q4_query = "What metric do we use to evaluate a search engine?"
r4 = ms_vector.search(model.encode(q4_query), num_results=1)
print(f"Q4 filename: {r4[0]['filename']}")


Q4 filename: 04-evaluation/lessons/05-search-metrics.md


In [6]:
# Q5
from minsearch import Index

ms_text = Index(text_fields=["content"], keyword_fields=["filename"])
ms_text.fit(chunks)

q5_query = "How do I store vectors in PostgreSQL?"

text_results = ms_text.search(query=q5_query, num_results=5)
vector_results = ms_vector.search(model.encode(q5_query), num_results=5)

text_files = {d["filename"] for d in text_results}
vector_files = {d["filename"] for d in vector_results}
only_vector = vector_files - text_files

print(f"vector top5: {vector_files}")
print(f"text top5: {text_files}")
print(f"Q5 in vector not text: {only_vector}")


vector top5: {'02-vector-search/lessons/08-pgvector.md', '03-orchestration/lessons/05-rag.md'}
text top5: {'02-vector-search/lessons/02-embeddings.md', '02-vector-search/lessons/01-intro.md', '03-orchestration/lessons/05-rag.md'}
Q5 in vector not text: {'02-vector-search/lessons/08-pgvector.md'}


In [7]:
# Q6 — run Q5 cell first (defines ms_text); fallback below if skipped
q6_query = "How do I give the model access to tools?"

if "ms_text" not in globals():
    from minsearch import Index

    ms_text = Index(text_fields=["content"], keyword_fields=["filename"])
    ms_text.fit(chunks)

text_results = ms_text.search(query=q6_query, num_results=5)
vector_results = ms_vector.search(model.encode(q6_query), num_results=5)


def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


fused = rrf([vector_results, text_results])
print(f"Q6 filename: {fused[0]['filename']}")


Q6 filename: 01-agentic-rag/lessons/13-function-calling.md
